# Baseline Evaluations for Custom Complex-Valued GNN Classifier Head

**Context:** We are evaluating a custom, complex-valued Graph Neural Network (GNN) functioning as a classifier head on top of frozen VGG16 features.

- **Input:** 25,088 flattened features (from a 7x7x512 VGG spatial grid)
- **Our Model:** ~250k parameters, dynamic edge routing, logarithmic magnitudes, phase superposition (~1.36 GFLOPs)
- **Dataset:** Imagenette (10-class subset of ImageNet)

**Goal:** Prove that our efficient, dynamic graph topology outperforms traditional, static architectures.

Each baseline cell below is **fully self-contained** — it loads its own data, defines its own model, trains (if applicable), evaluates, and prints results. You can run any cell independently.

**Note on labels:** The `.pt` data files store VGG16's soft predictions (softmax over 10 Imagenette classes) as `label`. For neural network baselines, we train with these soft targets using `CrossEntropyLoss` (matching our GNN training setup). For non-neural baselines, we use hard labels via `argmax`.

## Baseline 1: Linear Probing (Raw Feature Sanity Check)

**Architecture:** `nn.Linear(25088, 10)` — a single layer with no hidden units.

**Parameters:** ~250k (~250,890) | **FLOPs:** ~0.5 MegaFLOPs

**What it proves:** This matches our exact parameter budget. If our GNN outperforms this, it definitively proves that the complex graph routing, iterative message passing, and spatial topology extract more useful information than simple, flat linear weights (a straightforward dot product).

In [6]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from tqdm import tqdm

# ── Data Loading (identical to training.py) ──────────────────────────────────
THIS_DIR = Path(".").resolve()
DATA_DIR = THIS_DIR / "data"

train = torch.load(str(DATA_DIR / "imagenette_train_data.pt"), weights_only=False)
val   = torch.load(str(DATA_DIR / "imagenette_val_data.pt"),   weights_only=False)

x_train, y_train = train["data"], train["label"]  # y is soft distribution [N, 10]
x_val,   y_val   = val["data"],   val["label"]

BATCH_SIZE = 64
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(x_val,   y_val),   batch_size=BATCH_SIZE, shuffle=False)

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Train samples: {len(x_train)} | Val samples: {len(x_val)}")
print(f"Feature dim: {x_train.shape[1]} | Classes: {y_train.shape[1]}")

# ── Model ────────────────────────────────────────────────────────────────────
model = nn.Linear(25088, 10).to(device)
param_count = sum(p.numel() for p in model.parameters())
print(f"Linear Probe parameters: {param_count:,}")

# ── Training ─────────────────────────────────────────────────────────────────
EPOCHS = 20
LR = 1e-3
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print(f"\n{'='*60}")
print(f"Training Linear Probe | Epochs: {EPOCHS} | LR: {LR} | Batch: {BATCH_SIZE}")
print(f"{'='*60}")

for epoch in range(EPOCHS):
    model.train()
    train_loss, correct, total = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * y.size(0)
        correct += (logits.argmax(1) == y.argmax(1)).sum().item()
        total += y.size(0)

    train_loss /= total
    train_acc = 100.0 * correct / total

    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            val_loss += loss.item() * y.size(0)
            val_correct += (logits.argmax(1) == y.argmax(1)).sum().item()
            val_total += y.size(0)

    val_loss /= val_total
    val_acc = 100.0 * val_correct / val_total
    print(f"Epoch {epoch+1:>2}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

print(f"\n{'='*60}")
print(f"RESULT — Linear Probe: Val Accuracy = {val_acc:.2f}%")
print(f"{'='*60}")

Device: mps
Train samples: 9469 | Val samples: 3925
Feature dim: 25088 | Classes: 10
Linear Probe parameters: 250,890

Training Linear Probe | Epochs: 20 | LR: 0.001 | Batch: 64
Epoch  1/20 | Train Loss: 0.2158 | Train Acc: 95.25% | Val Loss: 0.1750 | Val Acc: 96.64%
Epoch  2/20 | Train Loss: 0.1973 | Train Acc: 98.22% | Val Loss: 0.2339 | Val Acc: 96.54%
Epoch  3/20 | Train Loss: 0.2467 | Train Acc: 98.25% | Val Loss: 0.3266 | Val Acc: 95.62%
Epoch  4/20 | Train Loss: 0.2671 | Train Acc: 98.65% | Val Loss: 0.3405 | Val Acc: 96.10%
Epoch  5/20 | Train Loss: 0.2781 | Train Acc: 98.81% | Val Loss: 0.3118 | Val Acc: 96.66%
Epoch  6/20 | Train Loss: 0.2976 | Train Acc: 98.62% | Val Loss: 0.3691 | Val Acc: 95.90%
Epoch  7/20 | Train Loss: 0.3119 | Train Acc: 98.69% | Val Loss: 0.3468 | Val Acc: 96.25%
Epoch  8/20 | Train Loss: 0.3230 | Train Acc: 98.88% | Val Loss: 0.3476 | Val Acc: 96.25%
Epoch  9/20 | Train Loss: 0.3213 | Train Acc: 98.76% | Val Loss: 0.4964 | Val Acc: 95.52%
Epoch 10/20 

## Baseline 2: Traditional ML — XGBoost

**Architecture:** Gradient-boosted decision tree ensemble (no neural network parameters).

**What it proves:** Establishes the inherent separability of the VGG features using purely statistical splitting. It answers: *"Are these features already so good that a basic statistical algorithm can classify them, or do they strictly require deep, topological reasoning to decode?"*

XGBoost is chosen over SVM because it handles high-dimensional (25,088-d) data more efficiently and tends to be a stronger out-of-the-box baseline for tabular/feature data.

In [3]:
import torch
import numpy as np
from pathlib import Path
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# ── Data Loading (identical to training.py) ──────────────────────────────────
THIS_DIR = Path(".").resolve()
DATA_DIR = THIS_DIR / "data"

train = torch.load(str(DATA_DIR / "imagenette_train_data.pt"), weights_only=False)
val   = torch.load(str(DATA_DIR / "imagenette_val_data.pt"),   weights_only=False)

x_train = train["data"].numpy()
y_train = train["label"].argmax(dim=1).numpy()  # hard labels for XGBoost
x_val   = val["data"].numpy()
y_val   = val["label"].argmax(dim=1).numpy()

print(f"Train samples: {len(x_train)} | Val samples: {len(x_val)}")
print(f"Feature dim: {x_train.shape[1]} | Classes: {len(np.unique(y_train))}")
print(f"Train label distribution: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Val   label distribution: {dict(zip(*np.unique(y_val,   return_counts=True)))}")

# ── XGBoost ──────────────────────────────────────────────────────────────────
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.3,   # sample 30% of 25k features per tree — critical for high-dim
    tree_method="hist",      # fast histogram-based method
    objective="multi:softmax",
    num_class=10,
    eval_metric="mlogloss",
    verbosity=1,
    random_state=42,
)

print(f"\n{'='*60}")
print(f"Training XGBoost | n_estimators: 300 | max_depth: 6")
print(f"{'='*60}")

xgb.fit(
    x_train, y_train,
    eval_set=[(x_val, y_val)],
    verbose=50,  # print every 50 rounds
)

# ── Evaluation ───────────────────────────────────────────────────────────────
train_preds = xgb.predict(x_train)
val_preds   = xgb.predict(x_val)

train_acc = 100.0 * accuracy_score(y_train, train_preds)
val_acc   = 100.0 * accuracy_score(y_val,   val_preds)

print(f"\nTrain Accuracy: {train_acc:.2f}%")
print(f"\n{'='*60}")
print(f"RESULT — XGBoost: Val Accuracy = {val_acc:.2f}%")
print(f"{'='*60}")
print(f"\nClassification Report (Validation):")
print(classification_report(y_val, val_preds, digits=4))

Train samples: 9469 | Val samples: 3925
Feature dim: 25088 | Classes: 10
Train label distribution: {np.int64(0): np.int64(964), np.int64(1): np.int64(954), np.int64(2): np.int64(987), np.int64(3): np.int64(862), np.int64(4): np.int64(948), np.int64(5): np.int64(954), np.int64(6): np.int64(966), np.int64(7): np.int64(931), np.int64(8): np.int64(944), np.int64(9): np.int64(959)}
Val   label distribution: {np.int64(0): np.int64(386), np.int64(1): np.int64(396), np.int64(2): np.int64(357), np.int64(3): np.int64(382), np.int64(4): np.int64(411), np.int64(5): np.int64(396), np.int64(6): np.int64(395), np.int64(7): np.int64(414), np.int64(8): np.int64(397), np.int64(9): np.int64(391)}

Training XGBoost | n_estimators: 300 | max_depth: 6
[0]	validation_0-mlogloss:1.98458
[50]	validation_0-mlogloss:0.23568
[100]	validation_0-mlogloss:0.14563
[150]	validation_0-mlogloss:0.12492
[200]	validation_0-mlogloss:0.11806
[250]	validation_0-mlogloss:0.11449
[299]	validation_0-mlogloss:0.11298

Train Accu

## Baseline 3: Spatial Flattening Test — GAP + Bottleneck MLP

**Architecture:** Global Average Pooling collapses the 7x7 spatial grid into a flat 512-d vector, followed by a 2-layer MLP: `Linear(512, 475) -> ReLU -> Linear(475, 10)`.

**Parameters:** ~248k (243,360 + 4,760 + 10 = 248,130)  | Matched to our GNN budget.

**What it proves:** GAP destroys local spatial relationships by averaging the entire 7x7 grid. Our GNN maintains arbitrary relationships across spatial nodes. If our GNN wins here, it proves that the graph's structural routing captures better relational data than simply averaging the image and running through a dense network.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path

# ── Data Loading (identical to training.py) ──────────────────────────────────
THIS_DIR = Path(".").resolve()
DATA_DIR = THIS_DIR / "data"

train = torch.load(str(DATA_DIR / "imagenette_train_data.pt"), weights_only=False)
val   = torch.load(str(DATA_DIR / "imagenette_val_data.pt"),   weights_only=False)

x_train, y_train = train["data"], train["label"]
x_val,   y_val   = val["data"],   val["label"]

BATCH_SIZE = 64
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(x_val,   y_val),   batch_size=BATCH_SIZE, shuffle=False)

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Train samples: {len(x_train)} | Val samples: {len(x_val)}")

# ── Model ────────────────────────────────────────────────────────────────────
class GAPBottleneckMLP(nn.Module):
    """
    Reshape flat 25088 features back to [B, 512, 7, 7], apply Global Average Pooling
    to get [B, 512], then run through a 2-layer MLP.
    """
    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(512, 475),
            nn.ReLU(),
            nn.Linear(475, 10),
        )

    def forward(self, x):
        # x: [B, 25088] -> reshape to VGG spatial grid -> GAP
        x = x.view(-1, 512, 7, 7)          # restore spatial structure
        x = x.mean(dim=(2, 3))              # Global Average Pooling -> [B, 512]
        return self.mlp(x)

model = GAPBottleneckMLP().to(device)
param_count = sum(p.numel() for p in model.parameters())
print(f"GAP + MLP parameters: {param_count:,}")

# ── Training ─────────────────────────────────────────────────────────────────
EPOCHS = 20
LR = 1e-3
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print(f"\n{'='*60}")
print(f"Training GAP + Bottleneck MLP | Epochs: {EPOCHS} | LR: {LR} | Batch: {BATCH_SIZE}")
print(f"{'='*60}")

for epoch in range(EPOCHS):
    model.train()
    train_loss, correct, total = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * y.size(0)
        correct += (logits.argmax(1) == y.argmax(1)).sum().item()
        total += y.size(0)

    train_loss /= total
    train_acc = 100.0 * correct / total

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            val_loss += loss.item() * y.size(0)
            val_correct += (logits.argmax(1) == y.argmax(1)).sum().item()
            val_total += y.size(0)

    val_loss /= val_total
    val_acc = 100.0 * val_correct / val_total
    print(f"Epoch {epoch+1:>2}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

print(f"\n{'='*60}")
print(f"RESULT — GAP + Bottleneck MLP: Val Accuracy = {val_acc:.2f}%")
print(f"{'='*60}")

## Baseline 4: Math Ablation — Standard GCN on Same Topology

**Architecture:** A standard Graph Convolutional Network (GCN) operating on the **exact same graph topology** as our model: 15,454 nodes, 3,136 input nodes, 256 output nodes, cardinality 200 (incoming edges per node), 5 message-passing iterations. The custom complex-valued phase/magnitude math is replaced with standard real-valued GCN propagation: `H' = ReLU(D⁻¹AHW + b)`.

**Parameters:** ~250k (tuned via hidden channel dimension). Implemented from scratch using sparse ops — no PyG dependency.

**What it proves:** The most scientifically rigorous ablation. It isolates our mathematical contribution — proving whether the specific phase/magnitude logic and logarithmic scaling are strictly superior to standard real-valued graph mathematics on the exact same topology.

**Note:** This baseline generates a random static edge structure matching the topology statistics (cardinality=200). The random seed is fixed for reproducibility. Training is slower than the other baselines due to sparse message passing over 15k+ nodes.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from tqdm import tqdm

# ── Data Loading (identical to training.py) ──────────────────────────────────
THIS_DIR = Path(".").resolve()
DATA_DIR = THIS_DIR / "data"

train = torch.load(str(DATA_DIR / "imagenette_train_data.pt"), weights_only=False)
val   = torch.load(str(DATA_DIR / "imagenette_val_data.pt"),   weights_only=False)

x_train, y_train = train["data"], train["label"]
x_val,   y_val   = val["data"],   val["label"]

# Smaller batch size — sparse message passing over 15k nodes is memory-intensive
BATCH_SIZE = 4
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(x_val,   y_val),   batch_size=BATCH_SIZE, shuffle=False)

# Use CPU for sparse ops — MPS sparse support is limited
device = "cpu"
print(f"Device: {device} (forced CPU — MPS/CUDA sparse support for this topology is unreliable)")
print(f"Train samples: {len(x_train)} | Val samples: {len(x_val)}")

# ── Graph Topology (matching run3/config.yaml) ───────────────────────────────
NUM_NODES    = 15454
INPUT_NODES  = 3136   # first 3136 nodes receive VGG features
OUTPUT_NODES = 256    # last 256 nodes are read out
CARDINALITY  = 200    # incoming edges per node
NUM_ITERS    = 5      # message-passing iterations (matches model.iterations)
VECTOR_DIM   = 8      # input feature dim per node (3136 * 8 = 25088)

# Build random static adjacency: each node gets 200 random incoming connections
# Normalized by 1/cardinality (row-normalized D^{-1}A)
torch.manual_seed(42)
sources = torch.randint(0, NUM_NODES, (NUM_NODES, CARDINALITY))
row_idx = torch.arange(NUM_NODES).unsqueeze(1).expand(NUM_NODES, CARDINALITY).reshape(-1)
col_idx = sources.reshape(-1)
values  = torch.full((NUM_NODES * CARDINALITY,), 1.0 / CARDINALITY)
adj = torch.sparse_coo_tensor(
    torch.stack([row_idx, col_idx]), values, (NUM_NODES, NUM_NODES)
).coalesce().to(device)

print(f"Graph: {NUM_NODES} nodes | {NUM_NODES * CARDINALITY:,} edges | {NUM_ITERS} iterations")
print(f"Adjacency: {adj.shape}, nnz={adj._nnz():,}")

# ── Model ────────────────────────────────────────────────────────────────────
class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x, adj):
        # GCN propagation: aggregate neighbors then transform
        # x: [N, in_dim], adj: sparse [N, N] (row-normalized)
        agg = torch.sparse.mm(adj, x)   # [N, in_dim]
        return self.linear(agg)          # [N, out_dim]


class GCNBaseline(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_iters, num_nodes, input_nodes, output_nodes):
        super().__init__()
        self.num_nodes = num_nodes
        self.input_nodes = input_nodes
        self.output_nodes = output_nodes
        self.hidden_dim = hidden_dim

        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.gcn_layers = nn.ModuleList([
            GCNLayer(hidden_dim, hidden_dim) for _ in range(num_iters)
        ])
        self.output_head = nn.Linear(output_nodes, 10)

    def forward(self, x, adj):
        # x: [B, 25088]
        B = x.size(0)
        results = []
        for b in range(B):
            # Initialize all node features to zero
            h = torch.zeros(self.num_nodes, self.hidden_dim, device=x.device)
            # Inject input features into input nodes
            inp = x[b].view(self.input_nodes, -1)        # [3136, 8]
            h[:self.input_nodes] = self.input_proj(inp)   # [3136, hidden_dim]

            # Iterative message passing
            for gcn in self.gcn_layers:
                h = torch.relu(gcn(h, adj))

            # Readout: take output nodes, mean-pool over hidden dim -> scalar per node
            out_feats = h[-self.output_nodes:]  # [256, hidden_dim]
            out = out_feats.mean(dim=1)         # [256]
            results.append(out)

        logits = self.output_head(torch.stack(results))  # [B, 10]
        return logits


# Compute hidden_dim to hit ~250k params:
#   input_proj: 8*h + h = 9h
#   5 GCN layers: 5*(h*h + h) = 5h^2 + 5h
#   output_head: 256*10 + 10 = 2570
#   Total = 5h^2 + 14h + 2570 ≈ 250k  =>  h ≈ 221
HIDDEN_DIM = 221

model = GCNBaseline(
    in_dim=VECTOR_DIM, hidden_dim=HIDDEN_DIM, num_iters=NUM_ITERS,
    num_nodes=NUM_NODES, input_nodes=INPUT_NODES, output_nodes=OUTPUT_NODES,
).to(device)

param_count = sum(p.numel() for p in model.parameters())
print(f"Standard GCN parameters: {param_count:,}")

# ── Training ─────────────────────────────────────────────────────────────────
EPOCHS = 20
LR = 1e-3
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print(f"\n{'='*60}")
print(f"Training Standard GCN | Epochs: {EPOCHS} | LR: {LR} | Batch: {BATCH_SIZE}")
print(f"Hidden dim: {HIDDEN_DIM} | Iterations: {NUM_ITERS}")
print(f"WARNING: This baseline is slow (~15k-node sparse GCN). Budget time accordingly.")
print(f"{'='*60}")

for epoch in range(EPOCHS):
    model.train()
    train_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for x, y in pbar:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x, adj)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * y.size(0)
        correct += (logits.argmax(1) == y.argmax(1)).sum().item()
        total += y.size(0)
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    train_loss /= total
    train_acc = 100.0 * correct / total

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x, adj)
            loss = criterion(logits, y)
            val_loss += loss.item() * y.size(0)
            val_correct += (logits.argmax(1) == y.argmax(1)).sum().item()
            val_total += y.size(0)

    val_loss /= val_total
    val_acc = 100.0 * val_correct / val_total
    print(f"Epoch {epoch+1:>2}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

print(f"\n{'='*60}")
print(f"RESULT — Standard GCN (same topology): Val Accuracy = {val_acc:.2f}%")
print(f"{'='*60}")

## Baseline 5: The Anchor — Original VGG16 Classifier Head

**Architecture:** The massive original VGG classifier: `Linear(25088, 4096) -> Linear(4096, 4096) -> Linear(4096, 10)`.

**Parameters:** 123.64 Million | **FLOPs:** 247 MegaFLOPs

**What it proves:** This is not for a direct apples-to-apples comparison, but to frame the ultimate value proposition. It allows us to conclude: *"We achieved comparable classification accuracy to the original architecture while stripping away 99.8% of the parameter footprint by utilizing dynamic graph routing."*

**Implementation note:** No training needed. The VGG16 predictions are already stored in the `.pt` data files as soft labels. We compare these against true Imagenette labels loaded from the downloaded dataset. (Ported from `eval_vgg_baseline.py`.)

In [1]:
import torch
import torchvision
from torchvision import transforms
from pathlib import Path
from sklearn.metrics import classification_report

# ── Data Loading ─────────────────────────────────────────────────────────────
THIS_DIR = Path(".").resolve()
DATA_DIR = THIS_DIR / "data"

# VGG16's stored predictions: soft targets [N, 10]
val = torch.load(str(DATA_DIR / "imagenette_val_data.pt"), weights_only=False)
vgg_preds = val["label"].argmax(dim=1)  # [N]
print(f"Val samples: {len(vgg_preds)}")

# True Imagenette labels — load dataset with same order (shuffle=False)
dataset = torchvision.datasets.Imagenette(
    root=str(DATA_DIR),
    split="val",
    size="320px",
    download=False,
    transform=transforms.ToTensor(),  # minimal transform just to iterate
)
true_labels = torch.tensor([dataset[i][1] for i in range(len(dataset))])

# ── Evaluation ───────────────────────────────────────────────────────────────
correct = (vgg_preds == true_labels).sum().item()
total = len(true_labels)
accuracy = 100.0 * correct / total

print(f"\nVGG16 original classifier head:")
print(f"  Parameters: ~123.64M")
print(f"  FLOPs:      ~247 MegaFLOPs")

print(f"\n{'='*60}")
print(f"RESULT — VGG16 Full Head: Val Accuracy = {correct}/{total} = {accuracy:.2f}%")
print(f"{'='*60}")

print(f"\nPer-class breakdown:")
class_names = dataset.classes if hasattr(dataset, 'classes') else [str(i) for i in range(10)]
print(classification_report(
    true_labels.numpy(), vgg_preds.numpy(),
    target_names=[str(c) for c in class_names] if class_names else None,
    digits=4,
))

Val samples: 3925

VGG16 original classifier head:
  Parameters: ~123.64M
  FLOPs:      ~247 MegaFLOPs

RESULT — VGG16 Full Head: Val Accuracy = 3902/3925 = 99.41%

Per-class breakdown:
                                                                  precision    recall  f1-score   support

                                        ('tench', 'Tinca tinca')     1.0000    0.9974    0.9987       387
                ('English springer', 'English springer spaniel')     0.9975    1.0000    0.9987       395
                                            ('cassette player',)     0.9916    0.9916    0.9916       357
                                       ('chain saw', 'chainsaw')     0.9895    0.9793    0.9844       386
                                   ('church', 'church building')     0.9927    0.9976    0.9951       409
                                         ('French horn', 'horn')     0.9924    0.9975    0.9949       394
                                   ('garbage truck', 'dustcart')     0.